# LRS TQQQ 策略研究 Notebook

> 目标：把「QQQ 200 日均线信号」从单一规则升级为可交易、可验证、可迭代的策略框架。

## 研究边界

- 标的：`TQQQ`（执行层）
- 主信号：`QQQ` 的 200 日均线（方向层）
- 宏观过滤：沿用当前项目的 6 个 Yahoo 代理指标
- 研究周期建议：最近 2 年为主，外加若干震荡窗口专项复盘

## 你当前关注的问题（转为研究问题）

1. 200MA 信号出现后，如何判定是否“有效信号”？
2. 宏观信号如何先行过滤，避免逆风硬上仓位？
3. 执行方式如何选：直接买入 TQQQ 还是卖 Put 接盘？
4. 仓位配比怎么定：一次满仓还是分层进场？
5. 横盘震荡阶段如何减少反复打脸？
6. 回测窗口多长合适（2 年 vs 5 年）？重点复盘哪些区间？


## 一、策略基本框架（V0.1）

将策略拆成 4 层：

1. **方向层（Direction）**：QQQ vs 200MA
2. **有效性层（Validation）**：信号是否可靠（避免假突破）
3. **执行层（Execution）**：直接买 TQQQ 或卖 Put 接盘
4. **风控层（Risk）**：仓位、止损、横盘降频

---

### 1) 方向层：基础信号

- 多头候选：`QQQ close > MA200`
- 空仓候选：`QQQ close < MA200`

这个层只给“方向”，不直接给“仓位大小/执行方式”。

### 2) 有效性层：通过后才进场

建议至少同时满足以下 3 条中的 2 条：

- **趋势确认**：QQQ 连续 N 日站上 MA200（建议 N=3~5）
- **斜率确认**：MA200 斜率为正（例如过去 20 日 MA200 变化 > 0）
- **波动确认**：短期波动率不过高（避免消息驱动尖刺后的回落）

### 3) 宏观过滤层（先行）

把 6 个宏观代理映射成 Risk-On / Risk-Off 分数：

- 美债收益率曲线（^IRX, ^FVX, ^TNX, ^TYX）
- 美元指数（DX-Y.NYB）
- 通胀预期代理（TIP）

建议先做一个简单规则：

- `macro_score >= +1`：允许做多
- `macro_score <= -1`：禁止新增多头，偏减仓/观望
- 介于中间：仅小仓位试错

### 4) 执行层（买入方式决策）

当方向+有效性+宏观全部通过后，再决定执行方式：

- 若价格离短均线不远、波动适中：**直接买 TQQQ**
- 若价格偏离较大（追高风险）或短期波动上升：**卖 Put 等接盘**

可先做离散选择器：

- `execution_mode = "direct_buy" | "sell_put"`

### 5) 风控层（仓位与震荡）

- 基础仓位分层：`25% -> 50% -> 75% -> 100%`
- 横盘识别（例如 ADX 低、ATR 收缩、布林带收敛）后：
  - 降低信号权重
  - 提高确认阈值
  - 最大仓位上限下调（例如从 100% 降到 50%）


## 二、重点风险点（必须重点回测）

### A. 假突破风险（Whipsaw）

- 价格短暂上穿 MA200 后快速跌回
- 典型后果：频繁开平仓、手续费和滑点吞噬收益
- 回测要看：
  - 单月交易次数
  - 连续亏损次数
  - 胜率变化 vs 收益变化

### B. 杠杆路径依赖风险（TQQQ 特有）

- 即使 QQQ 横盘，TQQQ 也可能因波动损耗表现变差
- 回测要看：
  - 横盘期净值回撤
  - 与 QQQ 基准收益差

### C. 宏观反身性风险

- 宏观信号变化往往先于价格，但有时“过早”
- 回测要看：
  - 宏观过滤是否显著降低回撤
  - 是否过度过滤导致错过主要上涨段

### D. 执行方式风险（直接买 vs 卖 Put）

- 卖 Put 在急跌中可能被动接刀
- 直接买在高波动追涨时容易回撤
- 回测要看：
  - 两种执行方式在不同波动 regime 下的收益/回撤
  - 行权/保证金占用（若接入真实期权数据）

### E. 仓位风险

- 一次性满仓对策略容错极低
- 分层建仓可能错过部分涨幅但更稳健
- 回测要看：
  - 收益回撤比（Calmar）
  - 最大单次回撤的可承受性


## 三、回测窗口与样本期建议

你的判断很合理：**先看 2 年更实用**。

- 2 年窗口优点：更贴近当前市场结构（利率、AI 主题、波动节奏）
- 5 年窗口优点：包含更多 regime，但会引入过时结构

建议采用“双层验证”：

1. **主回测窗口**：最近 2 年（策略调参主依据）
2. **稳健性窗口**：补充 1~2 个横盘/高波动子区间（不调参，仅验鲁棒）

### 横盘震荡重点复盘（方法）

先不预设具体日期，而是程序化识别：

- `QQQ` 的 20 日真实波动率低于分位阈值
- ADX 低于阈值（若你后续接 TA 指标）
- 价格在区间内来回穿越 MA200 次数高

在这些区间单独统计：

- 策略收益 vs 买入持有
- 交易频次
- 净值回撤
- 交易成本敏感性


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

QQQ = "QQQ"
TQQQ = "TQQQ"
MACRO_TICKERS = ["^IRX", "^FVX", "^TNX", "^TYX", "DX-Y.NYB", "TIP"]

START = (pd.Timestamp.today() - pd.DateOffset(years=2)).strftime("%Y-%m-%d")
END = pd.Timestamp.today().strftime("%Y-%m-%d")

print("backtest window:", START, "->", END)

In [ ]:
def load_price_series(ticker: str, start: str, end: str) -> pd.Series:
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty:
        raise ValueError(f"no data for {ticker}")
    return df["Close"].rename(ticker)


def load_macro_frame(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    cols = []
    for t in tickers:
        s = load_price_series(t, start, end)
        cols.append(s)
    out = pd.concat(cols, axis=1).sort_index()
    return out

In [ ]:
def compute_lrs_signal(qqq_close: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=qqq_close.index)
    df["close"] = qqq_close
    df["ma200"] = qqq_close.rolling(200).mean()
    df["ma200_slope_20"] = df["ma200"].diff(20)

    # 方向层：站上/跌破 200MA
    df["dir_long"] = (df["close"] > df["ma200"]).astype(int)

    # 有效性：连续站上 3 日 + ma200 斜率为正
    above = (df["close"] > df["ma200"]).astype(int)
    df["above_3d"] = (above.rolling(3).sum() == 3).astype(int)
    df["valid_trend"] = ((df["above_3d"] == 1) & (df["ma200_slope_20"] > 0)).astype(int)

    return df


def compute_macro_score(macro: pd.DataFrame) -> pd.Series:
    # 这是一个 V0 占位打分函数，后续应替换为更严谨的经济含义映射
    ret = macro.pct_change(5)
    score = pd.Series(0, index=macro.index, dtype=float)

    # 方向示例（仅研究起点）：
    # - 美元走强可能压制风险资产
    # - TIP 走强通常对风险偏好略正向
    if "DX-Y.NYB" in ret.columns:
        score += np.where(ret["DX-Y.NYB"] < 0, 1, -1)
    if "TIP" in ret.columns:
        score += np.where(ret["TIP"] > 0, 1, -1)

    return score.rename("macro_score")

In [ ]:
def position_sizing(valid_signal: pd.Series, macro_score: pd.Series) -> pd.Series:
    # 仓位规则 V0：
    # valid_signal=1 且 macro_score>=1 -> 100%
    # valid_signal=1 且 macro_score=0 -> 50%
    # 其余 -> 0%
    pos = pd.Series(0.0, index=valid_signal.index)
    pos[(valid_signal == 1) & (macro_score >= 1)] = 1.0
    pos[(valid_signal == 1) & (macro_score == 0)] = 0.5
    return pos.rename("target_position")


def backtest_tqqq(position: pd.Series, tqqq_close: pd.Series, fee_bps: float = 2.0) -> pd.DataFrame:
    ret = tqqq_close.pct_change().fillna(0.0)
    pos = position.reindex(ret.index).ffill().fillna(0.0)

    turnover = pos.diff().abs().fillna(0.0)
    fee = turnover * (fee_bps / 10000.0)

    strat_ret = pos.shift(1).fillna(0.0) * ret - fee
    eq = (1 + strat_ret).cumprod()
    bh = (1 + ret).cumprod()

    out = pd.DataFrame(
        {
            "ret_tqqq": ret,
            "position": pos,
            "turnover": turnover,
            "fee": fee,
            "ret_strategy": strat_ret,
            "equity_strategy": eq,
            "equity_buy_hold": bh,
        }
    )
    return out

In [ ]:
qqq = load_price_series(QQQ, START, END)
tqqq = load_price_series(TQQQ, START, END)
macro = load_macro_frame(MACRO_TICKERS, START, END)

sig = compute_lrs_signal(qqq)
score = compute_macro_score(macro)

panel = sig.join(score, how="left").ffill()
panel["target_position"] = position_sizing(panel["valid_trend"], panel["macro_score"])

bt = backtest_tqqq(panel["target_position"], tqqq)

panel.tail(), bt[["equity_strategy", "equity_buy_hold", "position"]].tail()

## 四、执行方式：直接买入 vs 卖 Put（研究设计）

你提到“当前价格偏高时，是否应卖 Put 接盘”非常关键。建议用“条件分流”来做：

### 条件分流规则（研究版）

- 当 `valid_trend=1` 且价格未明显偏离（例如 close 相对 MA20 的偏离不高）：`direct_buy`
- 当 `valid_trend=1` 但短期偏离过大/IV 偏高：`sell_put`

### 回测对比维度

1. 收益：年化、累计收益
2. 风险：最大回撤、下行波动
3. 稳定性：胜率、盈亏比、连续亏损
4. 成本：交易频次、手续费、期权滑点与行权影响

> 注意：卖 Put 研究需要期权链与成交假设，否则结论会偏乐观。


## 五、下一步落地计划（建议按顺序）

1. **先固化 V0 规则**：QQQ 200MA + 简单宏观过滤 + 分层仓位（不加期权）
2. **完成 2 年主回测**：输出收益/回撤/交易频次/换手
3. **专项复盘震荡窗口**：筛选 whipsaw 最严重区间，观察策略是否过度交易
4. **加入执行分流**：引入 direct_buy vs sell_put 的条件决策
5. **参数稳定性测试**：对确认天数、仓位阈值、宏观阈值做敏感性分析

---

### 关键原则

- 不追求“单期最高收益”，优先“跨区间可存活”
- 每增加一个过滤器，都要验证是否真正改善风险收益比
- 对 TQQQ 这类杠杆 ETF，回撤控制优先级高于信号数量


## 六、补充研究：LRS 原始定义与当前真实案例（2026-04）

### 1) LRS 的标准定义（Leverage for the Long Run）

LRS（Leverage for the Long Run Strategy）在本研究中的最小规则为：

| QQQ 与 200MA | 操作 |
|---|---|
| `QQQ 收盘 > 200MA` | 持有 TQQQ |
| `QQQ 收盘 < 200MA` | 清仓 TQQQ，转现金或短债 |

该规则的核心是“只跟随趋势，不做主观预测”。

### 2) 当前状态（教学样本）

基于你提供的信息：

- QQQ 近期处于约 `470-580` 区间波动
- QQQ 200 日均线约 `594`
- 当前 QQQ 低于 200MA

在 LRS 规则下，这对应**离场状态**（risk-off），并且非常适合作为“跌破后如何等待再入场”的研究样本。

### 3) 关键启发

- 同样是跌破 200MA，宏观背景不同，后续路径完全不同。
- 因此 LRS 在实盘中应升级为：
  - **均线给方向**
  - **宏观给强弱与等待时长**
  - **执行层决定买入方式与仓位节奏**


## 七、六个宏观指标在 LRS 中的操作化映射

你给出的分析非常关键：这些指标在 LRS 中不是用于“选股”，而是用于两件事：

1. **均线信号前预警**：是否提前降仓
2. **均线信号后定性**：这次跌破是浅跌还是深跌，回仓应快还是慢

### 指标作用与操作提示（研究版）

| 指标 | LRS中的主作用 | 偏多/顺风 | 偏空/逆风 |
|---|---|---|---|
| `IRX` | Fed政策方向/短端利率冲击 | 平稳或下行：可提高仓位 | 快速上行：提前降仓（如100%->60%） |
| `TNX` | 科技股贴现率锚 | 高位回落：重返均线上方后可积极回仓 | 持续上行：跌破后谨慎等待 |
| `TIP` | 实际利率压力 | 止跌/回升：风险偏好改善 | 持续下跌：实际利率上升，减仓 |
| `FVX/TYX` | 曲线形态与恢复速度 | 曲线正常化：跌破后恢复可能更快 | 倒挂/长期平坦：跌破后等待更久 |
| `DX-Y.NYB` | 全球流动性与盈利折算 | 走弱/平稳：利好成长风格 | 快速走强：压制风险资产 |

> 备注：`IRX` 在 LRS 里应被视为最高优先级观察项之一，`TNX/TIP` 为关键确认项。


## 八、LRS 宏观记分卡（V0.2）与仓位决策

### 1) 月初记分卡（可先手工）

对 5 组信号打分（顺风 +1，逆风 -1，中性 0）：

- `IRX`
- `TNX`
- `TIP`
- `Curve(FVX/TYX 与 IRX-TNX形态)`
- `DXY`

`macro_total = sum(scores)`

### 2) 均线上方时的仓位建议（持有期）

| 宏观顺风个数 | 建议仓位（TQQQ） | 说明 |
|---|---:|---|
| 4-5 | 100% | 让杠杆充分工作 |
| 2-3 | 60%-70% | 保留进攻，控制波动衰减 |
| 0-1 | 40%-50% 或更低 | 即便在均线上方也防守 |

### 3) 均线跌破后的回仓等待逻辑（离场期）

| 跌破时宏观状态 | 回仓节奏 |
|---|---|
| 顺风仍有 >=3 | 可能是情绪性下破；重新站回均线可较快回仓 |
| 逆风 >=4 | 可能是基本面驱动下跌；等待 IRX 与 TNX 同步改善再回仓 |

### 4) 执行分流（直接买 vs 卖Put）

在“允许回仓”的前提下：

- **直接买入**：价格偏离不大、波动中等、趋势确认充分
- **卖 Put 接盘**：价格偏高/波动偏大，偏向等更好成本入场

> 重要：卖 Put 需要期权链数据与成交假设，回测中必须纳入权利金、行权、保证金约束。


## 九、重点回测议程（基于你提出的问题）

建议把回测拆成 3 条主线，每条都做 2 年主窗口 + 震荡子区间专项复盘。

### 主线 A：有效信号过滤是否有价值

比较：

1. 纯 LRS（仅 QQQ vs 200MA）
2. LRS + 有效性过滤（连续站上N日、MA200斜率）
3. LRS + 有效性 + 宏观记分卡

观察：收益、最大回撤、交易次数、Calmar、横盘期表现。

### 主线 B：执行方式比较（直接买 vs 卖Put）

比较：

1. 全部 direct_buy
2. 条件分流 direct_buy/sell_put

观察：收益差、回撤差、资金占用、尾部风险（急跌时）。

### 主线 C：仓位管理鲁棒性

比较：

1. 满仓策略
2. 分层仓位策略（40/70/100）

观察：是否显著降低 TQQQ 在震荡期的净值磨损。

---

## 十、策略框架结论（当前版本）

你提出的方向非常正确：LRS 不应停留在“均线二元开关”。

当前建议的最小可交易框架是：

- **方向**：QQQ 200MA
- **过滤**：有效性 + 宏观记分卡
- **执行**：direct_buy / sell_put 分流
- **风控**：分层仓位 + 震荡降频

这套框架可直接作为策略页下一步实现蓝图：

1. 先做无期权版（direct_buy）
2. 再叠加期权执行分支
3. 最后做参数稳定性与样本外验证
